In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

with open('../input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print("total examples:", len(text))
print("vocab size:", vocab_size)
print(text[:25])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

block_size = 8
batch_size = 64
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(low=0, high=len(data)-block_size, size=(batch_size, ))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

Xb, Yb = get_batch('train')
print(Xb[0])
print(Yb[0])

cuda
total examples: 1115394
vocab size: 65
First Citizen:
Before we 
tensor([60, 63,  8,  1, 19, 53, 53, 42], device='cuda:0')
tensor([63,  8,  1, 19, 53, 53, 42,  1], device='cuda:0')


In [2]:
n_embd = 16
head_size = 1000
n_layer = 10

In [3]:
class FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd)
        )
    
    def forward(self, x):
        return x + self.net(x)
    
class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.headtoembd = nn.Linear(head_size, n_embd)
    
    def forward(self, x):
        B, T, C = x.shape

        q = self.query(x) 
        k = self.key(x)   # (B,T,hs)
        v = self.value(x)

        wei = q @ k.transpose(-1, -2) * head_size**(-0.5)
        tril = torch.tril(torch.ones((T, T), device=device))
        wei = wei.masked_fill(tril == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        out = wei @ v
        out = self.headtoembd(out)
        return x + out

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(n_embd),
            Head(),
            nn.LayerNorm(n_embd),
            FFN()
        )
    
    def forward(self, x):
        return self.net(x)

In [4]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ffdw = FFN()
        self.unembed = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        tok_emb = self.tok_emb(idx) # (B,T,C) -> (batch, block, embed)
        pos_emb = self.pos_emb(torch.arange(T, device=device)) # (block, embed)
        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ffdw(x)
        
        
        if targets is None:
            loss = None
            logits = x[:, [-1], :] # only need to calc last ones for generation
            logits = self.unembed(logits)
        else:
            logits = self.unembed(x)
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)    
    
        return logits, loss


    def generate(self, idx, max_tokens=1):
        for _ in range(max_tokens):
            logits, _ = self(idx[:,-block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, ix), dim=1) # lol no need to cut
        return idx

In [ ]:
max_iters = 10000
lr = 8e-4

model = GPT()
model.to(device)

# model.load_state_dict(state_dict=torch.load("v3.pt", weights_only=True))
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
for iter in range(max_iters):  
    Xb, Yb = get_batch('train')
    logits, loss = model(Xb, Yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % (max_iters // 5) == 0 or iter == max_iters - 1:
        print(loss.item())
torch.save(model.state_dict(), "v3.pt")
cont = torch.ones((1,1), dtype=torch.long, device=device)
print(decode(model.generate(cont, 100)[0].tolist()))

RuntimeError: Error(s) in loading state_dict for GPT:
	Missing key(s) in state_dict: "blocks.2.net.0.weight", "blocks.2.net.0.bias", "blocks.2.net.1.query.weight", "blocks.2.net.1.key.weight", "blocks.2.net.1.value.weight", "blocks.2.net.1.headtoembd.weight", "blocks.2.net.1.headtoembd.bias", "blocks.2.net.2.weight", "blocks.2.net.2.bias", "blocks.2.net.3.net.0.weight", "blocks.2.net.3.net.0.bias", "blocks.2.net.3.net.2.weight", "blocks.2.net.3.net.2.bias", "blocks.3.net.0.weight", "blocks.3.net.0.bias", "blocks.3.net.1.query.weight", "blocks.3.net.1.key.weight", "blocks.3.net.1.value.weight", "blocks.3.net.1.headtoembd.weight", "blocks.3.net.1.headtoembd.bias", "blocks.3.net.2.weight", "blocks.3.net.2.bias", "blocks.3.net.3.net.0.weight", "blocks.3.net.3.net.0.bias", "blocks.3.net.3.net.2.weight", "blocks.3.net.3.net.2.bias", "blocks.4.net.0.weight", "blocks.4.net.0.bias", "blocks.4.net.1.query.weight", "blocks.4.net.1.key.weight", "blocks.4.net.1.value.weight", "blocks.4.net.1.headto